<a href="https://colab.research.google.com/github/sergi-villanueva/Xatbot-1.4---Sergi-Villanueva/blob/main/XatBot_talent_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install google-generativeai requests beautifulsoup4 flask pyngrok flask-cors -q

In [ ]:
from google.colab import userdata
import google.generativeai as genai
import requests
from bs4 import BeautifulSoup
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import re

# =================== SECRETS ===================
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
genai.configure(api_key=GEMINI_API_KEY)

# Configuració per fer-lo més ràpid i net
model = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config={
        "temperature": 0.7,
        "max_output_tokens": 800,
    }
)

app = Flask(__name__)
CORS(app)

def clean_text(text):
    """Neteja el text de Markdown i formats estranys"""
    # Elimina ** i *
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    # Elimina salts de línia excessius
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

@app.route('/')
def home():
    return "✅ XatBot actiu!"

@app.route('/chat', methods=['POST'])
def chat():
    try:
        user_message = request.json.get('message', '') if request.json else ''

        WORDPRESS_URL = "https://svillanueva.inscastellbisbal.net"   # ← Canvia per la teva URL real

        # Scraping
        headers = {'User-Agent': 'Mozilla/5.0'}
        r = requests.get(WORDPRESS_URL, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')

        texts = [tag.get_text(strip=True) for tag in soup.find_all(['h1','h2','p','li','strong']) if len(tag.get_text(strip=True)) > 20]
        context = " ".join(texts[:40])

        prompt = f"""Ets un assistent professional, amable i directe de Sergi Villanueva.
        Respon sempre en català, de forma natural i sense usar Markdown (** o *).
        No facis llistes amb asteriscs. Escriu de forma fluida i conversacional.

        Informació de la web: {context}

        Pregunta: {user_message}"""

        response = model.generate_content(prompt)
        clean_reply = clean_text(response.text)

        return jsonify({"reply": clean_reply})

    except Exception as e:
        error_str = str(e).lower()
        if "429" in error_str or "quota" in error_str:
            return jsonify({"reply": "⏳ Estic amb molta demanda ara mateix. Torna-ho a provar d'aquí a 30-60 segons."})
        else:
            print(f"Error: {e}")
            return jsonify({"reply": "Ho sento, hi ha hagut un error. Torna-ho a provar."})

# ===================== INICIAR =====================
public_url = ngrok.connect(5000)
print("\n" + "="*60)
print("✅ XATBOT MILLORAT EN FUNCIONAMENT")
print(f"🔗 URL: {public_url}")
print("="*60)

app.run(port=5000)


✅ XATBOT MILLORAT EN FUNCIONAMENT
🔗 URL: NgrokTunnel: "https://unsatisfiable-unwhitewashed-despina.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 07:19:51] "OPTIONS /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 07:19:55] "POST /chat HTTP/1.1" 200 -
